# NB-02 | Benchmark: Baseline 2 — Character LoRA

Оценка **Anima Base + Turbo LoRA + Character LoRA** (@sltn trigger).  
Сравниваем с BL1 — насколько LoRA добавляет стиль Iro-toresu / Painterly.

**Особенность**: логируем metadata из safetensors файла Character LoRA.

In [1]:
import os, json, struct, hashlib
from pathlib import Path
import torch, numpy as np, mlflow
from PIL import Image
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
import open_clip
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [2]:
BASELINE_ID   = 'baseline_2'
BASELINE_DESC = 'Anima Base + Turbo LoRA + Character LoRA (trigger: @sltn)'

GENERATED_DIR  = Path('generated/baseline_2')
REFERENCE_DIR  = Path('reference/characters')

# Путь до файла Character LoRA для чтения metadata
# Если файл не найден — metadata будет пустой, это не критично
CHAR_LORA_PATH = Path('ComfyUI/models/loras/Anima/sltn_character_v2.safetensors')

MLFLOW_URI      = 'http://127.0.0.1:5000'
EXPERIMENT_NAME = 'anima_baseline_benchmarks'

MODEL_PARAMS = {
    'model.name':          'anima_base',
    'model.family':        'cosmos2_dit',
    'model.type':          'character_lora',
    'model.diffusion':     'animaBase_v1.safetensors',
    'model.text_encoder':  'qwen_3_06b_base.safetensors',
    'model.vae':           'qwen_image_vae.safetensors',
    'model.char_lora':     'sltn_character_v2.safetensors',
    'model.lora_stack':    'turbo+character',
    'model.trigger':       '@sltn',
    'model.dataset_size':  1172,
    'model.dataset_type':  'synthetic_bootstrapped',
    'model.dataset_bias':  '1girl_solo_outdoors',
    'model.caption_type':  'booru_tags_wd14_moat',
    'model.metadata_source': 'config_known',
}

INFER_PARAMS = {
    'infer.baseline_id':    BASELINE_ID,
    'infer.sampler':        'euler_ancestral',
    'infer.scheduler':      'normal',
    'infer.steps':          12,
    'infer.cfg':            2.0,
    'infer.width':          512,
    'infer.height':         512,
    'infer.turbo_strength': 1.0,
    'infer.style_strength': 0.85,
    'infer.llm_expansion':  False,
    'infer.seed_policy':    'idx*7919',
    'infer.prompt_set':     'char_prompts_v1',
    'infer.n_images':       100,
}

DATA_PARAMS = {
    'data.generated_dir':  str(GENERATED_DIR),
    'data.reference_dir':  str(REFERENCE_DIR),
    'data.reference_type': 'character_splash_art',
}

print('Config loaded ✓')

Config loaded ✓


In [3]:
def load_images_as_tensors(folder, size=(299,299), limit=None):
    files = sorted(folder.glob('*.png'))
    if limit: files = files[:limit]
    return torch.stack([torch.tensor(np.array(Image.open(p).convert('RGB').resize(size))).permute(2,0,1) for p in files])

def load_images_pil(folder, size=None, limit=None):
    files = sorted(folder.glob('*.png'))
    if limit: files = files[:limit]
    return [Image.open(p).convert('RGB').resize(size) if size else Image.open(p).convert('RGB') for p in files]

def make_grid(images, n=8, thumb=128):
    imgs = [img.resize((thumb,thumb)) for img in images[:n]]
    row = Image.new('RGB', (thumb*len(imgs), thumb), (20,20,20))
    for i,im in enumerate(imgs): row.paste(im, (thumb*i, 0))
    return row

def read_safetensors_metadata(path):
    try:
        with open(path, 'rb') as f:
            n = struct.unpack('<Q', f.read(8))[0]
            return json.loads(f.read(n).decode('utf-8')).get('__metadata__', {})
    except Exception as e:
        return {'error': str(e)}

print('Helpers ready ✓')

Helpers ready ✓


In [4]:
def compute_kid(gen_dir, ref_dir, subset=50):
    kid = KernelInceptionDistance(subset_size=subset, normalize=True).to(DEVICE)
    kid.update(load_images_as_tensors(ref_dir).to(DEVICE), real=True)
    kid.update(load_images_as_tensors(gen_dir).to(DEVICE), real=False)
    m, s = kid.compute()
    return float(m.cpu()), float(s.cpu())

def compute_clip_score(gen_dir, prompts):
    model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
    model = model.to(DEVICE).eval()
    tokenizer = open_clip.get_tokenizer('ViT-B-32')
    scores = []
    for i, p in enumerate(tqdm(sorted(gen_dir.glob('*.png')), desc='CLIP')):
        if i >= len(prompts): break
        img = preprocess(Image.open(p).convert('RGB')).unsqueeze(0).to(DEVICE)
        txt = tokenizer([prompts[i]]).to(DEVICE)
        with torch.no_grad():
            a = model.encode_image(img); a /= a.norm(dim=-1,keepdim=True)
            b = model.encode_text(txt);  b /= b.norm(dim=-1,keepdim=True)
            scores.append(float((a*b).sum().cpu()))
    return float(np.mean(scores))

def compute_lpips_diversity(gen_dir, n_pairs=200):
    lpips_fn = LearnedPerceptualImagePatchSimilarity(net_type='vgg', normalize=True).to(DEVICE)
    files = sorted(gen_dir.glob('*.png'))
    idx = np.random.default_rng(42).integers(0, len(files), size=(n_pairs,2))
    scores = []
    def to_t(p):
        return torch.tensor(np.array(Image.open(p).convert('RGB').resize((256,256))).astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(DEVICE)
    for a,b in tqdm(idx, desc='LPIPS'):
        if a==b: continue
        with torch.no_grad(): scores.append(float(lpips_fn(to_t(files[a]), to_t(files[b])).cpu()))
    return float(np.mean(scores))

print('Metric functions ready ✓')

Metric functions ready ✓


In [5]:
BASE_PROMPTS = [
    '1girl, solo, long hair, blue eyes, smile, outdoors, sky',
    '1girl, solo, short hair, red eyes, serious, city background',
    '1girl, solo, twin tails, green eyes, school uniform, outdoors',
    '1girl, solo, white hair, purple eyes, simple background',
    '1girl, solo, ponytail, brown eyes, jacket, forest',
    '1boy, solo, short hair, blue eyes, armor, standing',
    '1girl, solo, black hair, dress, indoors',
    '1girl, solo, cat ears, pink hair, happy, outdoors',
    '1boy, solo, messy hair, red scarf, city',
    '1girl, solo, silver hair, golden eyes, cape, sky background',
]
PROMPTS_100 = [BASE_PROMPTS[i % 10] for i in range(100)]
print(f'Prompts: {len(PROMPTS_100)}')

Prompts: 100


In [6]:
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

run_name = f'eval_{BASELINE_ID}_charLoRA_cfg{INFER_PARAMS["infer.cfg"]}_w{INFER_PARAMS["infer.style_strength"]}'

with mlflow.start_run(run_name=run_name) as run:
    print(f'Run ID: {run.info.run_id}')

    mlflow.set_tags({
        'evaluation_type': 'benchmark',
        'training_logged_retroactively': 'true',
        'family':          'character',
        'baseline_id':     BASELINE_ID,
        'environment':     'local_rtx4060ti_16gb',
        'framework':       'comfyui',
    })

    mlflow.log_params(MODEL_PARAMS)
    mlflow.log_params(INFER_PARAMS)
    mlflow.log_params(DATA_PARAMS)

    # Читаем metadata из safetensors если файл существует
    if CHAR_LORA_PATH.exists():
        lora_meta = read_safetensors_metadata(CHAR_LORA_PATH)
        # Сохраняем полный metadata как artifact
        Path('tmp').mkdir(parents=True, exist_ok=True)
        meta_path = f'tmp/{BASELINE_ID}_lora_metadata.json'
        with open(meta_path, 'w') as f:
            json.dump(lora_meta, f, indent=2)
        mlflow.log_artifact(meta_path, artifact_path='model_provenance')
        # Ключевые поля логируем как params с префиксом train.
        train_fields = [
            'ss_network_dim', 'ss_network_alpha', 'ss_learning_rate',
            'ss_unet_lr', 'ss_text_encoder_lr', 'ss_epoch',
            'ss_steps', 'ss_optimizer', 'ss_batch_size',
            'ss_training_comment', 'ss_resolution'
        ]
        for k in train_fields:
            if k in lora_meta:
                mlflow.log_param(f'train.{k.replace("ss_","")}', lora_meta[k])
        print(f'LoRA metadata logged: {list(lora_meta.keys())[:5]}...')
    else:
        print('LoRA file not found for metadata extraction — skipping')

    kid_mean, kid_std = compute_kid(GENERATED_DIR, REFERENCE_DIR)
    clip_score        = compute_clip_score(GENERATED_DIR, PROMPTS_100)
    lpips_div         = compute_lpips_diversity(GENERATED_DIR)

    mlflow.log_metrics({
        'eval.kid_mean':        kid_mean,
        'eval.kid_std':         kid_std,
        'eval.clip_score':      clip_score,
        'eval.lpips_diversity': lpips_div,
        'eval.n_generated':     float(len(list(GENERATED_DIR.glob('*.png')))),
        'eval.n_reference':     float(len(list(REFERENCE_DIR.glob('*.png')))),
    })

    grid = make_grid(load_images_pil(GENERATED_DIR, limit=8))
    gp = f'tmp/{BASELINE_ID}_grid.png'; grid.save(gp)
    mlflow.log_artifact(gp, artifact_path='samples')

    print(f'\n=== BL2 Results ===')
    print(f'KID:   {kid_mean:.6f} ± {kid_std:.6f}')
    print(f'CLIP:  {clip_score:.4f}')
    print(f'LPIPS: {lpips_div:.4f}')
    print(f'✓ Logged to MLflow')

Run ID: 77c7508c567244dfa162bdb70c6feec9
LoRA file not found for metadata extraction — skipping


e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Metric `Kernel Inception Distance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
LPIPS: 100%|██████████| 200/200 [01:03<00:00,  3.15it/s]


=== BL2 Results ===
KID:   0.042832 ± 0.004296
CLIP:  0.2993
LPIPS: 0.6917
✓ Logged to MLflow
🏃 View run eval_baseline_2_charLoRA_cfg2.0_w0.85 at: http://127.0.0.1:5000/#/experiments/1/runs/77c7508c567244dfa162bdb70c6feec9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
